In [19]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

import pyswarm

import pandas as pd
import numpy as np

In [16]:
data = pd.read_csv('./dataset/USR.csv')
data['date'] = data['Year'].astype(str) + '-' + data['Month'].astype(str)
data.set_index('date', inplace=True)

In [3]:
def create_lags(df, k, value_col):
    """
    Transforma a série temporal em um DataFrame supervisionado.

    k: quantidade de lags
    value_col: nome da coluna com os valores numéricos
    """

    data = df[value_col].copy()

    result = pd.DataFrame()

    for i in range(k, 0, -1):
        result[f'x-{i}'] = data.shift(i)

    result['x'] = data

    result = result.dropna().reset_index(drop=True)

    return result

In [ ]:
def fitness(params, df, value_col, test_size=0.2):
    """
    Fitness para otimização dos hiperparâmetros de uma MLP
    utilizando PSO.

    params[0]: quantidade de lags (k)
    params[1]: neurônios na camada escondida (n)

    Retorna o MSE de validação.
    """

    k = int(round(params[0]))
    n = int(round(params[1]))

    # Transformação supervisionada
    data = create_lags(df, k, value_col)

    X = data.drop(columns=['x']).values
    y = data['x'].values

    # Divisão cronológica
    split = int(len(X) * (1 - test_size))

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Normalização
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train = scaler_X.fit_transform(X_train)
    X_test = scaler_X.transform(X_test)

    y_train = scaler_y.fit_transform(
        y_train.reshape(-1, 1)
    ).ravel()

    # Arquitetura da MLP
    model = MLPRegressor(
        hidden_layer_sizes=(n,),
        activation='relu',
        solver='adam',
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=30
    )

    # Treinamento
    model.fit(X_train, y_train)

    # Previsão
    predictions = model.predict(X_test)

    # Retorna à escala original
    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).ravel()

    # Erro de validação
    mse = mean_squared_error(y_test, predictions)

    return mse

In [ ]:
# Limites inferiores e superiores
lb = np.array([1, 1])
ub = np.array([30, 100])

bounds = (lb, ub)


# Função adaptadora para o PySwarms
def objective_function(particles):

    costs = []

    for particle in particles:

        cost = fitness(
            particle,
            df=df,
            value_col='valor'
        )

        costs.append(cost)

    return np.array(costs)


# Configurações do PSO
options = {
    'c1': 1.5,
    'c2': 1.5,
    'w': 0.7
}

optimizer = ps.single.GlobalBestPSO(
    n_particles=30,
    dimensions=2,
    options=options,
    bounds=bounds
)


# Executa otimização
best_cost, best_position = optimizer.optimize(
    objective_function,
    iters=100
)


# Recupera os melhores hiperparâmetros
best_k = int(round(best_position[0]))
best_n = int(round(best_position[1]))

print("Melhor MSE:", best_cost)
print("Melhor quantidade de lags:", best_k)
print("Melhor quantidade de neurônios:", best_n)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(optimizer.cost_history)

plt.xlabel('Iteração')
plt.ylabel('Melhor MSE')
plt.title('Convergência do PSO')

plt.grid()
plt.show()